# GGNN — Edge Classification Benchmark
## UNSW IoT Botnet · 5-Class Edge-Level Classification

## Section 0 — Configuration (only path you must change)

In [1]:
# -- CHANGE THIS -------------------------------------------------------------
GRAPH_PT_PATH = "/content/drive/MyDrive/botnet_graph.pt"
# ----------------------------------------------------------------------------

SEED = 42

# -- GGNN Hyperparameters -------------------------------------------------
# NOTE: num_layers = GRU unroll steps inside GatedGraphConv, NOT the
# number of stacked GNN layers. Different semantic than GCN/GAT.
HP = dict(
    hidden_dim          = 64,
    num_layers          = 3,
    dropout             = 0.3,
    lr                  = 5e-4,
    weight_decay        = 1e-4,
    batch_size          = 2048,
    fan_out             = [10, 10, 10],
    max_epochs          = 100,
    patience            = 15,
    min_delta           = 5e-5,
    val_ratio           = 0.10,
    label_smoothing     = 0.03,
)
print('Hyperparameters:', HP)


Hyperparameters: {'hidden_dim': 64, 'num_layers': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 2048, 'fan_out': [10, 10, 10], 'max_epochs': 100, 'patience': 15, 'min_delta': 5e-05, 'val_ratio': 0.1, 'label_smoothing': 0.03}


## Section 1 — Install Dependencies

In [2]:

import sys, subprocess, re

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", *args])

# Remove any previously installed PyG packages that can conflict.
subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch-geometric", "pyg-lib", "torch-scatter",
    "torch-sparse", "torch-cluster", "torch-spline-conv"
])

import torch

torch_ver = re.match(r"(\d+\.\d+\.\d+)", torch.__version__).group(1)
cuda_ver = torch.version.cuda

if cuda_ver is None:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
else:
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace('.', '')}.html"

pip_install(
    "torch-geometric",
    "pyg-lib",
    "torch-scatter",
    "torch-sparse",
    "torch-cluster",
    "torch-spline-conv",
    "-f", wheel_url
)

print("Installed PyG dependencies from:", wheel_url)


Installed PyG dependencies from: https://data.pyg.org/whl/torch-2.11.0+cu128.html


## Section 2 — Imports & Reproducibility

In [3]:

import os, time, warnings
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GatedGraphConv

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


Device: cuda


## Section 3 — Load Graph

In [4]:

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

data = torch.load(GRAPH_PT_PATH, map_location='cpu', weights_only=False)
print(data)

print(f"\nNodes            : {data.num_nodes}")
print(f"Edges (total)    : {data.edge_index.shape[1]:,}")
print(f"Node feat dim    : {data.x.shape[1]}")
print(f"Edge feat dim    : {data.edge_attr.shape[1]}")
print(f"Num classes      : {data.num_classes}")

NUM_CLASSES   = int(data.num_classes)
NODE_FEAT_DIM = int(data.x.shape[1])
EDGE_FEAT_DIM = int(data.edge_attr.shape[1])

# Expected label order for this benchmark (alphabetical — matches LabelEncoder output).
LABEL_MAP = {0: 'DDoS', 1: 'DoS', 2: 'Normal', 3: 'Reconnaissance', 4: 'Theft'}

# Build a fallback lookup from (src, dst) -> original edge row index.
# This is used only if the loader does not expose input_id.
edge_pair_to_id = {}
for eid, (s, d) in enumerate(data.edge_index.t().tolist()):
    edge_pair_to_id.setdefault((int(s), int(d)), int(eid))
print(f"Built edge pair lookup for {len(edge_pair_to_id):,} unique directed pairs.")


Mounted at /content/drive
Data(x=[93, 6], edge_index=[2, 3668615], edge_attr=[3668615, 15], edge_label=[3668615], train_mask=[3668615], test_mask=[3668615], num_nodes=93, num_classes=5)

Nodes            : 93
Edges (total)    : 3,668,615
Node feat dim    : 6
Edge feat dim    : 15
Num classes      : 5
Built edge pair lookup for 249 unique directed pairs.


## Section 4 — Train / Validation / Test Split

In [5]:

all_train_edge_idx = data.train_mask.nonzero(as_tuple=False).view(-1).long()
test_edge_idx      = data.test_mask.nonzero(as_tuple=False).view(-1).long()

if hasattr(data, 'val_mask') and data.val_mask is not None and int(data.val_mask.sum()) > 0:
    val_edge_idx   = data.val_mask.nonzero(as_tuple=False).view(-1).long()
    train_edge_idx = all_train_edge_idx
    print("Using existing validation mask.")
else:
    # Stratified split from the original training edges only.
    train_labels_np = data.edge_label[all_train_edge_idx].cpu().numpy()
    rel_indices = np.arange(len(all_train_edge_idx))
    tr_rel, val_rel = train_test_split(
        rel_indices,
        test_size=HP['val_ratio'],
        random_state=SEED,
        stratify=train_labels_np,
    )
    train_edge_idx = all_train_edge_idx[tr_rel]
    val_edge_idx   = all_train_edge_idx[val_rel]
    print(f"Created stratified validation split with val_ratio={HP['val_ratio']:.2f}.")

def describe_split(name, edge_idx):
    labels = data.edge_label[edge_idx]
    counts = torch.bincount(labels, minlength=NUM_CLASSES)
    total = int(edge_idx.numel())
    txt = ", ".join([f"{LABEL_MAP[i]}={int(counts[i])}" for i in range(NUM_CLASSES)])
    print(f"{name:<6} edges: {total:>10,} | {txt}")

describe_split("Train", train_edge_idx)
describe_split("Val",   val_edge_idx)
describe_split("Test",  test_edge_idx)


Created stratified validation split with val_ratio=0.10.
Train  edges:  2,641,335 | DDoS=1387183, DoS=1188133, Normal=333, Reconnaissance=65627, Theft=59
Val    edges:    293,482 | DDoS=154132, DoS=132015, Normal=37, Reconnaissance=7292, Theft=6
Test   edges:    733,705 | DDoS=385309, DoS=330112, Normal=107, Reconnaissance=18163, Theft=14


## Section 5 — Loss (unweighted)


In [6]:
# Unweighted loss — no inverse-frequency correction
# (class_weights removed to study the impact on minority-class recall)

criterion = nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])
print(f"Loss: CrossEntropyLoss(label_smoothing={HP['label_smoothing']})")


Loss: CrossEntropyLoss(label_smoothing=0.03)


## Section 6 — DataLoaders

> **Fix applied (v2)**: `make_loader` now accepts `mp_edge_idx` to restrict
> the message-passing graph to training edges only during train and val.
> This eliminates structural leakage where test-set edges could appear in
> neighborhood sampling context while computing node embeddings.
> The test loader still uses the full graph (training complete, no gradients flow).

In [7]:
def make_loader(edge_idx, batch_size, shuffle, mp_edge_idx=None):
    """
    Build a LinkNeighborLoader.

    mp_edge_idx : indices of edges used for message-passing (MP graph).
                  Pass train_edge_idx for train/val loaders so test edges
                  never appear in the MP neighborhood during training
                  (eliminates structural leakage).
                  Pass None for the test loader — training is complete,
                  no gradients flow, so full-graph sampling is safe.
    """
    edge_idx = edge_idx.long()
    if mp_edge_idx is not None:
        mp_edge_index = data.edge_index[:, mp_edge_idx.long()]
    else:
        mp_edge_index = data.edge_index

    from torch_geometric.data import Data as PygData
    mp_data = PygData(
        x=data.x,
        edge_index=mp_edge_index,
        edge_attr=data.edge_attr,
        num_nodes=data.num_nodes,
    )

    return LinkNeighborLoader(
        mp_data,
        num_neighbors=HP['fan_out'],
        edge_label_index=data.edge_index[:, edge_idx],
        edge_label=data.edge_label[edge_idx].long(),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
    )

# Train loader  : MP graph contains only training edges.
# Val loader    : val edges scored but not in the MP graph.
# Test loader   : training done, full graph safe (no gradient flow).


def resolve_labeled_edge_attr(batch, seed_edge_idx, full_edge_attr, pair_lookup):
    """
    Fetch edge attributes for the labeled (seed) edges in this batch.

    Preferred: batch.input_id -> seed_edge_idx -> global edge row.
    Fallback : (src, dst) pair lookup (used if input_id is absent).
    """
    if hasattr(batch, "input_id") and batch.input_id is not None:
        input_id = batch.input_id.detach().cpu().long()
        seed_edge_idx_cpu = seed_edge_idx.detach().cpu().long()
        global_edge_ids = seed_edge_idx_cpu[input_id]
        return full_edge_attr[global_edge_ids].to(batch.x.device)

    edge_label_index = batch.edge_label_index
    if hasattr(batch, "n_id"):
        if int(edge_label_index.max()) < int(batch.n_id.numel()):
            src = batch.n_id[edge_label_index[0].detach().cpu().long()]
            dst = batch.n_id[edge_label_index[1].detach().cpu().long()]
        else:
            src = edge_label_index[0].detach().cpu().long()
            dst = edge_label_index[1].detach().cpu().long()
    else:
        src = edge_label_index[0].detach().cpu().long()
        dst = edge_label_index[1].detach().cpu().long()

    edge_ids = []
    for s, d in zip(src.tolist(), dst.tolist()):
        eid = pair_lookup.get((int(s), int(d)))
        if eid is None:
            raise KeyError(f"Could not resolve edge attribute for pair ({s}, {d}).")
        edge_ids.append(eid)

    edge_ids = torch.tensor(edge_ids, dtype=torch.long)
    return full_edge_attr[edge_ids].to(batch.x.device)

print("Edge-attribute resolver ready.")


Edge-attribute resolver ready.


## Section 7 — GGNN Model Definition

In [8]:

class GGNNEdgeClassifier(nn.Module):
    """
    GGNN edge-level classifier.

    Node encoder   : Linear projection
    GGNN encoder    : GatedGraphConv message passing
    Edge scorer     : MLP([h_src || h_dst || edge_attr]) -> class logits
    """
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.dropout = dropout

        self.node_encoder = nn.Linear(node_feat_dim, hidden_dim)
        self.ggnn = GatedGraphConv(hidden_dim, num_layers)
        self.norm = nn.LayerNorm(hidden_dim)

        mlp_in = hidden_dim * 2 + edge_feat_dim

        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        h = self.ggnn(h, edge_index)
        h = self.norm(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def forward(self, x, edge_index, edge_attr, edge_label_index):
        h = self.encode_nodes(x, edge_index)
        src, dst = edge_label_index
        edge_repr = torch.cat([h[src], h[dst], edge_attr], dim=-1)
        return self.edge_mlp(edge_repr)




## Section 9 — Training & Validation Loop

> **Fix applied (v2)**: Early stopping now uses **val_f1 as the single criterion**.
> The previous dual-criterion logic reset patience when either val_loss or val_f1
> improved, keeping training alive on negligible deltas past the true optimum.
> Single-criterion stopping is cleaner and consistent with standard practice.

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device, seed_edge_idx):
    model.train()
    total_loss, total_correct, total_edges = 0.0, 0, 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        edge_attr = resolve_labeled_edge_attr(
            batch=batch,
            seed_edge_idx=seed_edge_idx,
            full_edge_attr=data.edge_attr,
            pair_lookup=edge_pair_to_id,
        )

        out = model(
            batch.x,
            batch.edge_index,
            edge_attr,
            batch.edge_label_index,
        )
        labels = batch.edge_label.long()
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        preds = out.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_loss    += loss.item() * labels.size(0)
        total_edges   += labels.size(0)

    return total_loss / total_edges, total_correct / total_edges


@torch.no_grad()
def evaluate(model, loader, criterion, device, seed_edge_idx):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    total_edges = 0

    for batch in loader:
        batch = batch.to(device)

        edge_attr = resolve_labeled_edge_attr(
            batch=batch,
            seed_edge_idx=seed_edge_idx,
            full_edge_attr=data.edge_attr,
            pair_lookup=edge_pair_to_id,
        )

        out = model(
            batch.x,
            batch.edge_index,
            edge_attr,
            batch.edge_label_index,
        )
        labels = batch.edge_label.long()
        loss   = criterion(out, labels)

        total_loss  += loss.item() * labels.size(0)
        total_edges += labels.size(0)
        all_preds.append(out.argmax(dim=-1).cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    avg_loss = total_loss / total_edges
    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, macro_f1, all_preds, all_labels


## Section — Multi-Seed Evaluation (added)

In [10]:
# ============================================================
# Section 14 — Multi-Seed Evaluation (GGNN, bot_iot, unweighted_loss)
# ============================================================
import copy, json, os
import numpy as np
import pandas as pd

MODEL_CLASS  = GGNNEdgeClassifier
MODEL_TAG    = "GGNN"
DATASET_TAG  = "bot_iot"
STRATEGY_TAG = "unweighted_loss"

SEEDS = [0, 1]

def build_criterion(train_edge_idx_r):
    train_labels_r = data.edge_label[train_edge_idx_r].long()
    counts_r = torch.bincount(train_labels_r, minlength=NUM_CLASSES).float().clamp(min=1)
    return nn.CrossEntropyLoss(label_smoothing=HP['label_smoothing'])

def run_once(seed):
    set_seed(seed)

    if hasattr(data, 'val_mask') and data.val_mask is not None and int(data.val_mask.sum()) > 0:
        val_edge_idx_r   = data.val_mask.nonzero(as_tuple=False).view(-1).long()
        train_edge_idx_r = all_train_edge_idx
    else:
        train_labels_np = data.edge_label[all_train_edge_idx].cpu().numpy()
        rel_indices = np.arange(len(all_train_edge_idx))
        tr_rel, val_rel = train_test_split(
            rel_indices, test_size=HP['val_ratio'],
            random_state=seed, stratify=train_labels_np,
        )
        train_edge_idx_r = all_train_edge_idx[tr_rel]
        val_edge_idx_r   = all_train_edge_idx[val_rel]

    criterion_r = build_criterion(train_edge_idx_r)

    train_loader_r = make_loader(train_edge_idx_r, HP['batch_size'],     shuffle=True,  mp_edge_idx=train_edge_idx_r)
    val_loader_r   = make_loader(val_edge_idx_r,   HP['batch_size'] * 2, shuffle=False, mp_edge_idx=train_edge_idx_r)
    test_loader_r  = make_loader(test_edge_idx,    HP['batch_size'] * 2, shuffle=False, mp_edge_idx=None)

    model_r = MODEL_CLASS(
        node_feat_dim = NODE_FEAT_DIM,
        edge_feat_dim = EDGE_FEAT_DIM,
        hidden_dim    = HP['hidden_dim'],
        num_layers    = HP['num_layers'],
        num_classes   = NUM_CLASSES,
        dropout       = HP['dropout'],
    ).to(DEVICE)

    optimizer_r = torch.optim.Adam(model_r.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    scheduler_r = ReduceLROnPlateau(optimizer_r, mode='min', factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1, patience_count, best_state = -float('inf'), 0, None
    history = []
    for epoch in range(1, HP['max_epochs'] + 1):
        set_seed(seed + epoch)
        train_loss, train_acc = train_one_epoch(model_r, train_loader_r, optimizer_r, criterion_r, DEVICE, train_edge_idx_r)
        val_loss, val_acc, val_f1, _, _ = evaluate(model_r, val_loader_r, criterion_r, DEVICE, val_edge_idx_r)
        scheduler_r.step(val_loss)
        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                             val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

        if val_f1 > best_val_f1 + HP['min_delta']:
            best_val_f1 = val_f1
            patience_count = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model_r.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= HP['patience']:
            break

    model_r.load_state_dict(best_state)
    _, test_acc, test_macro_f1, y_pred, y_true = evaluate(model_r, test_loader_r, criterion_r, DEVICE, test_edge_idx)

    test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    test_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0)

    per_class_f1        = f1_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    per_class_recall    = recall_score(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))

    per_class = {
        LABEL_MAP[i]: dict(f1=float(per_class_f1[i]), precision=float(per_class_precision[i]), recall=float(per_class_recall[i]))
        for i in range(NUM_CLASSES)
    }

    return dict(
        seed=seed, epochs_ran=epoch,
        acc=float(test_acc), macro_f1=float(test_macro_f1),
        precision=float(test_precision), recall=float(test_recall),
        per_class=per_class,
        history=history,
    )

results = [run_once(s) for s in SEEDS]

training_logs = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS,
    per_seed_history={r['seed']: r.pop('history') for r in results},
)

def agg_mean_std(vals):
    arr = np.array(vals, dtype=float)
    return dict(mean=float(arr.mean()), std=float(arr.std(ddof=1)) if len(arr) > 1 else 0.0)

overall = {metric: agg_mean_std([r[metric] for r in results]) for metric in ['acc', 'macro_f1', 'precision', 'recall']}

per_class_agg = {}
for cls in LABEL_MAP.values():
    per_class_agg[cls] = {metric: agg_mean_std([r['per_class'][cls][metric] for r in results]) for metric in ['f1', 'precision', 'recall']}

summary = dict(
    dataset=DATASET_TAG, strategy=STRATEGY_TAG, model=MODEL_TAG,
    seeds=SEEDS, n_seeds=len(SEEDS),
    per_seed_results=results, overall=overall, per_class=per_class_agg,
)

print(f"\n{'='*60}")
print(f"  {MODEL_TAG} — {DATASET_TAG} / {STRATEGY_TAG} — {len(SEEDS)}-seed summary")
print(f"{'='*60}")
for metric, stat in overall.items():
    print(f"  {metric:<10}: {stat['mean']:.4f} ± {stat['std']:.4f}")
print("\n  Per-class F1 (mean ± std):")
for cls, stat in per_class_agg.items():
    f1s = stat['f1']
    print(f"    {cls:<16}: {f1s['mean']:.4f} ± {f1s['std']:.4f}")

out_dir = "/content/results" if os.path.isdir("/content") else "."
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/{DATASET_TAG}_{STRATEGY_TAG}_{MODEL_TAG}_multiseed.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: {out_path}")

logs_path = f"{out_dir}/{DATASET_TAG}_{STRATEGY_TAG}_{MODEL_TAG}_trainlogs.json"
with open(logs_path, "w") as f:
    json.dump(training_logs, f, indent=2)
print(f"Saved: {logs_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
    colab_files.download(logs_path)
except ImportError:
    pass



  GGNN — bot_iot / unweighted_loss — 2-seed summary
  acc       : 0.9982 ± 0.0000
  macro_f1  : 0.5960 ± 0.0001
  precision : 0.5970 ± 0.0000
  recall    : 0.5951 ± 0.0002

  Per-class F1 (mean ± std):
    DDoS            : 0.9987 ± 0.0001
    DoS             : 0.9985 ± 0.0000
    Normal          : 0.0000 ± 0.0000
    Reconnaissance  : 0.9830 ± 0.0005
    Theft           : 0.0000 ± 0.0000

Saved: /content/results/bot_iot_unweighted_loss_GGNN_multiseed.json
Saved: /content/results/bot_iot_unweighted_loss_GGNN_trainlogs.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>